In [75]:
# Carregando as bibliotecas
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import joblib
import json

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

In [76]:
# Carregando datasets
df_initial = pd.read_csv(
    "../datasets/processed/df_initial_v1.csv"
)

df_final_v1 = pd.read_csv(
    "../datasets/processed/df_united_v1.csv"
)

In [77]:
print("Initial:", df_initial.shape)
print("Final V1:", df_final_v1.shape)

Initial: (10000, 23)
Final V1: (115934, 23)


In [78]:
df_final_train, df_final_holdout = train_test_split(
    df_final_v1,
    test_size=0.20,
    stratify=df_final_v1["label"],
    random_state=42
)

In [79]:
print("df_final_train :", len(df_final_train))
print("df_final_holdout :", len(df_final_holdout))

df_final_train : 92747
df_final_holdout : 23187


In [80]:
df_3a2 = pd.concat(
    [df_initial, df_final_train],
    ignore_index=True
)

In [81]:
print(df_3a2.shape)

df_3a2["label"].value_counts()

(102747, 23)


label
1    51374
0    51373
Name: count, dtype: int64

In [82]:
X_train = df_3a2.drop(
    columns=["label"],
    errors="ignore"
)

y_train = df_3a2["label"]

In [83]:
X_test = df_final_holdout.drop(
    columns=["label", "source"],
    errors="ignore"
)

y_test = df_final_holdout["label"]

In [84]:
print(X_train.shape)
print(X_test.shape)

(102747, 22)
(23187, 22)


In [85]:
top12 = [
    "HostnameLength",
    "DomainInPaths",
    "DomainInSubdomains",
    "SubdomainLevel",
    "PathLength",
    "NumDash",
    "PathLevel",
    "UrlLength",
    "NumDashInHostname",
    "NumDots",
    "NumNumericChars",
    "QueryLength"
]

In [86]:
X_train_top12 = X_train[top12]
X_test_top12 = X_test[top12]

In [87]:
rf_top12 = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_top12.fit(
    X_train_top12,
    y_train
)

rf_pred = rf_top12.predict(X_test_top12)

In [88]:
dt_top12 = DecisionTreeClassifier(
    random_state=42
)

dt_top12.fit(
    X_train_top12,
    y_train
)

dt_pred = dt_top12.predict(X_test_top12)

In [89]:
lr_top12 = LogisticRegression(
    max_iter=1000
)

lr_top12.fit(
    X_train_top12,
    y_train
)

lr_pred = lr_top12.predict(X_test_top12)

In [90]:
def evaluate_model(name, y_true, y_pred):

    print(f"\n===== {name} =====")

    print(
        "Accuracy:",
        accuracy_score(y_true, y_pred)
    )

    print(
        "Precision:",
        precision_score(y_true, y_pred)
    )

    print(
        "Recall:",
        recall_score(y_true, y_pred)
    )

    print(
        "F1:",
        f1_score(y_true, y_pred)
    )

In [91]:
evaluate_model(
    "Random Forest",
    y_test,
    rf_pred
)

evaluate_model(
    "Decision Tree",
    y_test,
    dt_pred
)

evaluate_model(
    "Logistic Regression",
    y_test,
    lr_pred
)


===== Random Forest =====
Accuracy: 0.9959460042265063
Precision: 0.9988720173535792
Recall: 0.9930130251013543
F1: 0.9959339043169825

===== Decision Tree =====
Accuracy: 0.9870617156165092
Precision: 0.9972699251431087
Recall: 0.9767963426205469
F1: 0.9869269653128813

===== Logistic Regression =====
Accuracy: 0.9833958683745202
Precision: 0.9766116686511311
Recall: 0.9905115155697404
F1: 0.9835124834054216


In [92]:
cm = confusion_matrix(
    y_test,
    rf_pred
)

cm

array([[11581,    13],
       [   81, 11512]])

In [93]:
print(
    classification_report(
        y_test,
        rf_pred
    )
)

              precision    recall  f1-score   support

           0       0.99      1.00      1.00     11594
           1       1.00      0.99      1.00     11593

    accuracy                           1.00     23187
   macro avg       1.00      1.00      1.00     23187
weighted avg       1.00      1.00      1.00     23187



In [94]:
results = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(y_test, rf_pred),
        "Precision": precision_score(y_test, rf_pred),
        "Recall": recall_score(y_test, rf_pred),
        "F1": f1_score(y_test, rf_pred)
    },
    {
        "Model": "Decision Tree",
        "Accuracy": accuracy_score(y_test, dt_pred),
        "Precision": precision_score(y_test, dt_pred),
        "Recall": recall_score(y_test, dt_pred),
        "F1": f1_score(y_test, dt_pred)
    },
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(y_test, lr_pred),
        "Precision": precision_score(y_test, lr_pred),
        "Recall": recall_score(y_test, lr_pred),
        "F1": f1_score(y_test, lr_pred)
    }
])

results

,Model,Accuracy,Precision,Recall,F1
0,Random Forest,0.995946,0.998872,0.993013,0.995934
1,Decision Tree,0.987062,0.997270,0.976796,0.986927
2,Logistic Regression,0.983396,0.976612,0.990512,0.983512


In [95]:
import os
joblib.dump(rf_top12, os.path.join("../models/scenario_3a2", "random_forest.pkl"))
joblib.dump(dt_top12, os.path.join("../models/scenario_3a2", "decision_tree.pkl"))
joblib.dump(lr_top12, os.path.join("../models/scenario_3a2", "logistic_regression.pkl"))

['../models/scenario_3a2\\logistic_regression.pkl']

In [96]:
joblib.dump(list(X_train_top12), os.path.join("../artifacts/scenario_3a2", "feature_columns.pkl"))

['../artifacts/scenario_3a2\\feature_columns.pkl']

In [97]:
results.to_csv(
    os.path.join("../artifacts/scenario_3a2", "metrics_scenario_3a2.csv"),
    index=False
)

O aumento de desempenho no Scenario 3A não está diretamente associado à complexidade do modelo ou ao número de features, mas à qualidade dos dados utilizados. O Scenario 3A2 demonstra que um subconjunto pequeno de features estruturais é suficiente para manter performance quase equivalente.